# Description

In this notebook, we prepare the Korns symbolic regression dataset for benchmarking of our algorithm. The dataset comprises 15 symbolic expression. Each expression contains a subset of five independent variables. Each independent variable is drawn uniformly from the interval [-50, 50]. For each expression we draw 10000 data points.

In [11]:
import numpy as np
import sympy as sp
import h5py
from dataclasses import dataclass
from typing import Callable, Dict, Tuple

In [2]:
@dataclass(frozen=True)
class KornsProblem:
    name: str
    expr: sp.Expr
    func: Callable[[np.ndarray], np.ndarray]

In [3]:
def _sympy_vars() -> Tuple[sp.Symbol, sp.Symbol, sp.Symbol, sp.Symbol, sp.Symbol]:
    x0, x1, x2, x3, x4 = sp.symbols("x0 x1 x2 x3 x4", real=True)
    return x0, x1, x2, x3, x4

In [4]:
def _build_korns_problems() -> Dict[str, KornsProblem]:
    x0, x1, x2, x3, x4 = _sympy_vars()

    exprs: Dict[str, sp.Expr] = {
        "P1":  sp.Float("1.57") + sp.Float("24.3") * x3,
        "P2":  sp.Float("0.23") + sp.Float("14.2") * ((x3 + x1) / (sp.Float("3.0") * x4)),
        "P3":  sp.Float("-5.41") + sp.Float("4.9") * (((x3 - x0) + (x1 / x4)) / (sp.Integer(3) * x4)),
        "P4":  sp.Float("-2.3") + sp.Float("0.13") * sp.sin(x2),
        "P5":  sp.Float("3.0") + sp.Float("2.13") * sp.log(x4),
        "P6":  sp.Float("1.3") + sp.Float("0.13") * sp.sqrt(x0),
        "P7":  sp.Float("213.80940889") * (sp.Integer(1) - sp.exp(-sp.Float("0.54723748542") * x0)),
        "P8":  sp.Float("6.87") + sp.Float("11") * sp.sqrt(sp.Float("7.23") * x0 * x3 * x4),
        "P9":  (sp.sqrt(x0) / sp.log(x1)) * (sp.exp(x2) / (x3**2)),
        "P10": sp.Float("0.81") + sp.Float("24.3") * ((sp.Integer(2)*x1 + sp.Integer(3)*(x2**2)) /
                                                      (sp.Integer(4)*(x3**3) + sp.Integer(5)*(x4**4))),
        "P11": sp.Float("6.87") + sp.Float("11") * sp.cos(sp.Float("7.23") * (x0**3)),
        "P12": sp.Float("2.0") - sp.Float("2.1") * (sp.cos(sp.Float("9.8") * x0) * sp.sin(sp.Float("1.3") * x4)),
        "P13": sp.Float("32.0") - sp.Float("3.0") * ((sp.tan(x0)/sp.tan(x1)) * (sp.tan(x2)/sp.tan(x3))),
        "P14": sp.Float("22.0") - sp.Float("4.2") * ((sp.cos(x0) - sp.tan(x1)) * (sp.tanh(x2)/sp.sin(x3))),
        "P15": sp.Float("12.0") - sp.Float("6.0") * ((sp.tan(x0)/sp.exp(x1)) * (sp.log(x2) - sp.tan(x3))),
    }

    problems: Dict[str, KornsProblem] = {}
    for name, expr in exprs.items():
        f_np = sp.lambdify((x0, x1, x2, x3, x4), expr, modules="numpy")

        def make_func(f):
            def _eval(X: np.ndarray) -> np.ndarray:
                return np.asarray(f(X[:, 0], X[:, 1], X[:, 2], X[:, 3], X[:, 4]), dtype=float)
            return _eval

        problems[name] = KornsProblem(name=name, expr=expr, func=make_func(f_np))

    return problems

In [5]:
KORNS_PROBLEMS: Dict[str, KornsProblem] = _build_korns_problems()

In [6]:
def generate_korns_dataset(
    problem_id: str,
    n_samples: int = 10_000,
    x_low: float = -50.0,
    x_high: float = 50.0,
    y_abs_max: float = 100.0,
    seed: int = 0,
    batch_size: int = 50_000,
) -> Tuple[np.ndarray, np.ndarray, sp.Expr]:
    prob = KORNS_PROBLEMS[problem_id]
    rng = np.random.default_rng(seed)

    X_keep = []
    y_keep = []

    remaining = n_samples
    while remaining > 0:
        Xb = rng.uniform(x_low, x_high, size=(batch_size, 5))

        with np.errstate(all="ignore"):
            yb = prob.func(Xb)

        yb = np.asarray(yb).reshape(-1)

        mask = np.isfinite(yb) & (np.abs(yb) <= y_abs_max)
        Xv = Xb[mask]
        yv = yb[mask]

        if Xv.shape[0] == 0:
            continue

        take = min(remaining, Xv.shape[0])
        X_keep.append(Xv[:take])
        y_keep.append(yv[:take])
        remaining -= take

    X = np.vstack(X_keep).astype(np.float64, copy=False)
    y = np.concatenate(y_keep).astype(np.float64, copy=False)

    if X.shape != (n_samples, 5):
        raise RuntimeError(f"Internal error: X has shape {X.shape}, expected {(n_samples, 5)}")
    if y.shape != (n_samples,):
        raise RuntimeError(f"Internal error: y has shape {y.shape}, expected {(n_samples,)}")

    return X, y, prob.expr

In [7]:
def generate_all_korns_datasets(
    n_samples: int = 10_000,
    seed: int = 0,
    **kwargs,
) -> Dict[str, Dict[str, object]]:
    out: Dict[str, Dict[str, object]] = {}
    for k, pid in enumerate(sorted(KORNS_PROBLEMS.keys(), key=lambda s: int(s[1:]))):
        X, y, expr = generate_korns_dataset(
            pid,
            n_samples=n_samples,
            seed=seed+k,
            **kwargs,
        )
        out[pid] = {"X": X, "y": y, "expr": expr}
    return out

In [8]:
datasets = generate_all_korns_datasets(seed=0)

# Save dataset

In [12]:
def save_korns_hdf5(datasets: dict, path: str):
    with h5py.File(path, "w") as f:
        for pid, data in datasets.items():
            grp = f.create_group(pid)

            grp.create_dataset("X", data=data["X"], compression="gzip")
            grp.create_dataset("y", data=data["y"], compression="gzip")

            # Save symbolic expression as string
            grp.attrs["expr_str"] = str(data["expr"])
            grp.attrs["expr_srepr"] = sp.srepr(data["expr"])

def load_korns_hdf5(path: str):
    out = {}
    with h5py.File(path, "r") as f:
        for pid in f.keys():
            grp = f[pid]
            X = grp["X"][:]
            y = grp["y"][:]

            expr = sp.sympify(grp.attrs["expr_str"])
            out[pid] = {"X": X, "y": y, "expr": expr}
    return out

In [13]:
save_korns_hdf5(datasets, "korns_dataset.hdf5")